In [2]:
# MSP430FR5043 firmware update

Use the USB connection widget below to upload TI-TXT, Intel HEX, or `.mspfw` firmware. The Python updater API also supports Wi-Fi. See the [update guide](../fw/esp32/docs/msp430_update.md) for wiring, ESP32 partition requirements, and recovery.


'\n   Copyright (C) Sergei Vostrikov (Sergio5714), 2026\n   \n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n       http://www.apache.org/licenses/LICENSE-2.0\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n\n   SPDX-License-Identifier: Apache-2.0\n'

## Before programming

Stop acquisition and close other clients or serial monitors. Connect MSP430 TEST, RESET, and all four JTAG signals as described in the guide; SPI alone is insufficient. Disconnect an external MSP-FET.

Select a COM port and click **Open**, select the firmware file, then click **Upload and program**. Commit reboots the ESP32 and temporarily interrupts connectivity while JTAG programming runs. Live progress is unavailable during this boot-time stage. After reconnection, **Check status** retrieves the saved result and diagnostics.

`COMPLETE` confirms programming and verification, not successful application startup. Reconnect the acquisition client, apply configuration, and verify normal acquisition afterward. Do not disconnect power during programming.


In [1]:
import ipywidgets as widgets
from IPython.display import display
from wulpus.usb_cdc_link import WulpusProUsbCdcLink
from wulpus.msp430_update import MSP430Updater

link = WulpusProUsbCdcLink()
port = widgets.Dropdown(description='COM port:', options=[])
refresh = widgets.Button(description='Refresh ports', icon='refresh')
open_port = widgets.Button(description='Open', button_style='info')
disconnect = widgets.Button(description='Disconnect', disabled=True)
connection = widgets.HTML(value='<b>Status:</b> Not connected')
update_panel = widgets.Output()

def refresh_ports(_=None):
    devices = link.get_available()
    port.options = [(str(device), device) for device in devices]
    open_port.disabled = not devices or link.connected
    connection.value = ('<b>Status:</b> Select a COM port' if devices else
                        '<b>Status:</b> No WULPUS PRO USB device found')

def open_selected_port(_):
    selected = port.value
    if selected is None:
        return
    update_panel.clear_output()
    if not link.open(selected):
        connection.value = f'<b>Status:</b> Failed to open {selected.device}'
        return
    connection.value = f'<b>Status:</b> Connected to {selected.device}'
    open_port.disabled = True
    disconnect.disabled = False
    with update_panel:
        MSP430Updater(link).show_widget()

def disconnect_port(_):
    if link.connected:
        link.close()
    update_panel.clear_output()
    connection.value = '<b>Status:</b> Disconnected'
    open_port.disabled = port.value is None
    disconnect.disabled = True

refresh.on_click(refresh_ports)
open_port.on_click(open_selected_port)
disconnect.on_click(disconnect_port)
display(widgets.VBox([widgets.HBox([port, refresh, open_port, disconnect]), connection, update_panel]))
refresh_ports()